# EEG Seizure Detection: Baseline Models

This notebook trains and compares classical baseline classifiers for EEG seizure detection:
- **Linear/probabilistic baselines**: Logistic Regression, Naive Bayes
- **Kernel and instance-based baselines**: SVM, KNN
- **Tree-based baselines**: Random Forest, XGBoost, LightGBM


## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project to path
cur_dir = Path(os.getcwd())
proj_dir = cur_dir.resolve().parent.parent
sys.path.insert(0, str(proj_dir))

# Import models
from utils.models.model_classes import (
    LogisticRegressionModel,
    NaiveBayesModel,
    SVMModel,
    RandomForestModel,
    KNNModel,
    XGBModel,
    LGBModel,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
)

print("✓ Imports successful")

## 2. Load Data

In [ ]:
# Define data paths
SPLIT_PATH = proj_dir / "data" / "split" / "task_1"
SELECTED_FILES = ["chb04_08", "chb04_05", "chb04_01", "chb04_02"]

# Load data from multiple subjects
def load_subject_data(subject_name, file_name):
    """Load data from a specific subject file."""
    file_path = SPLIT_PATH / subject_name / f"{file_name}.npz"
    data = np.load(file_path)
    
    return {
        'X_train': data['X_train'],
        'y_train': data['y_train'].astype(int),
        'X_val': data['X_val'],
        'y_val': data['y_val'].astype(int),
        'X_test': data['X_test'],
        'y_test': data['y_test'].astype(int),
        'feature_names': data['feature_names'],
        'channels': data['channels'],
    }

# Combine data from multiple files
all_data = {}
for file_name in SELECTED_FILES:
    subject = file_name.split('_')[0]
    try:
        data = load_subject_data(subject, file_name)
        all_data[file_name] = data
    except Exception as e:
        print(f"Warning: Could not load {file_name}: {e}")

# Use the first file whose training split contains both classes.
# Fall back to the first available file if none does.
def has_two_classes(split_data):
    return len(np.unique(split_data['y_train'])) > 1

if all_data:
    file_name, data = next(
        ((name, item) for name, item in all_data.items() if has_two_classes(item)),
        next(iter(all_data.items())),
    )

    X_train, y_train = data['X_train'], data['y_train']
    X_val, y_val = data['X_val'], data['y_val']
    X_test, y_test = data['X_test'], data['y_test']

    print(f"✓ Loaded data from {file_name}")
    print(f"  Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"  Train class distribution: {np.bincount(y_train, minlength=2)}")
    print(f"  Val class distribution: {np.bincount(y_val, minlength=2)}")
    print(f"  Test class distribution: {np.bincount(y_test, minlength=2)}")

## 3. Create DataFrame for Models

Convert numpy arrays to DataFrames for compatibility with model preprocessing methods.

In [ ]:
feature_names = data['feature_names']

# Create DataFrames
df_train = pd.DataFrame(X_train, columns=feature_names)
df_train['is_seizure'] = y_train

df_val = pd.DataFrame(X_val, columns=feature_names)
df_val['is_seizure'] = y_val

df_test = pd.DataFrame(X_test, columns=feature_names)
df_test['is_seizure'] = y_test

print("✓ DataFrames created")
print(f"  Train: {df_train.shape}")
print(f"  Val: {df_val.shape}")
print(f"  Test: {df_test.shape}")

## 4. Baseline Models Training

Train and evaluate baseline classifiers.

In [ ]:
# Dictionary to store results
results = {}
models_dict = {}

# Define baseline models
baseline_models = [
    ('LogisticRegression', LogisticRegressionModel(model_name='LogisticRegression')),
    ('NaiveBayes', NaiveBayesModel(model_name='NaiveBayes')),
    ('SVM', SVMModel(model_name='SVM', kernel='rbf')),
    ('KNN', KNNModel(model_name='KNN', n_neighbors=5)),
    ('RandomForest', RandomForestModel(model_name='RandomForest', n_estimators=100, max_depth=20)),
]

print("Training baseline models...\n")

for name, model in baseline_models:
    print(f"Training {name}...")
    try:
        # Preprocess data
        X_train_proc, y_train_proc = model.preprocess(df_train, is_training=True, verbose=False)
        X_val_proc, y_val_proc = model.preprocess(df_val, is_training=False, verbose=False)
        X_test_proc, y_test_proc = model.preprocess(df_test, is_training=False, verbose=False)
        
        # Train
        model.train(X_train_proc, y_train_proc)
        
        # Predict
        y_pred = model.predict(X_test_proc)
        
        # Get probabilities if available
        try:
            y_proba = model.predict_proba(X_test_proc)
            roc_auc = roc_auc_score(y_test_proc, y_proba)
        except:
            roc_auc = np.nan
        
        # Compute metrics
        acc = accuracy_score(y_test_proc, y_pred)
        prec = precision_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        
        results[name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'roc_auc': roc_auc,
        }
        models_dict[name] = model
        
        print(f"  ✓ Accuracy: {acc:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}\n")
    except Exception as e:
        print(f"  ✗ Error: {e}\n")
        results[name] = {}

## 5. Gradient Boosting Models

In [ ]:
# Gradient boosting models
boosting_models = [
    ('XGBoost', XGBModel(model_name='XGBoost', n_estimators=100, max_depth=7, tree_method='hist')),
    ('LightGBM', LGBModel(model_name='LightGBM', n_estimators=100, max_depth=7)),
]

print("Training gradient boosting models...\n")

for name, model in boosting_models:
    print(f"Training {name}...")
    try:
        # Preprocess data
        X_train_proc, y_train_proc = model.preprocess(df_train, is_training=True, verbose=False)
        X_val_proc, y_val_proc = model.preprocess(df_val, is_training=False, verbose=False)
        X_test_proc, y_test_proc = model.preprocess(df_test, is_training=False, verbose=False)
        
        # Train
        model.train(X_train_proc, y_train_proc)
        
        # Predict
        y_pred = model.predict(X_test_proc)
        
        # Get probabilities if available
        try:
            y_proba = model.predict_proba(X_test_proc)[:, 1]
            roc_auc = roc_auc_score(y_test_proc, y_proba)
        except:
            roc_auc = np.nan
        
        # Compute metrics
        acc = accuracy_score(y_test_proc, y_pred)
        prec = precision_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test_proc, y_pred, average='weighted', zero_division=0)
        
        results[name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'roc_auc': roc_auc,
        }
        models_dict[name] = model
        
        print(f"  ✓ Accuracy: {acc:.4f}, F1: {f1:.4f}, ROC-AUC: {roc_auc:.4f}\n")
    except Exception as e:
        print(f"  ✗ Error: {e}\n")
        results[name] = {}

## 6. Results Comparison


In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('f1', ascending=False)

print("\n" + "="*80)
print("MODEL COMPARISON RESULTS")
print("="*80)
print(results_df.to_string())
print("="*80)

## 7. Visualization of Results


In [ ]:
# Plot comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['accuracy', 'precision', 'recall', 'f1']
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for metric, (i, j) in zip(metrics, positions):
    ax = axes[i, j]
    if metric in results_df.columns:
        results_df[metric].plot(kind='barh', ax=ax, color='steelblue')
        ax.set_xlabel(metric.capitalize())
        ax.set_title(f'{metric.capitalize()} Comparison')
        ax.set_xlim([0, 1])

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Comparison plot saved as 'model_comparison.png'")

## 8. Best Model Analysis


In [ ]:
# Find best model
best_model_name = results_df['f1'].idxmax()
best_model = models_dict[best_model_name]

print(f"\nBest Model: {best_model_name}")
print(f"F1 Score: {results_df.loc[best_model_name, 'f1']:.4f}")
print(f"ROC-AUC: {results_df.loc[best_model_name, 'roc_auc']:.4f}")
print(f"Accuracy: {results_df.loc[best_model_name, 'accuracy']:.4f}")
print(f"Precision: {results_df.loc[best_model_name, 'precision']:.4f}")
print(f"Recall: {results_df.loc[best_model_name, 'recall']:.4f}")

## 9. Summary and Recommendations

### Key Findings:
1. **Simple baselines** provide interpretable reference performance.
2. **Tree-based baselines** such as Random Forest, XGBoost, and LightGBM are useful stronger tabular baselines.
3. Metrics should be interpreted together with the class distribution of each split, because some EEG files can contain only one class.

### Recommendations:
- Start model comparisons from these baselines before adding more complex architectures.
- Prefer balanced and event-aware splits for final evaluation.
- Perform cross-validation and hyperparameter tuning after the splitting strategy is stable.
- Monitor performance across multiple subjects/files, not only a single recording.
